# THPTQG Trends — Google Colab

Phan tich diem THPT quoc gia **2021-2025** va du bao xu huong.

**Chay lan luot tu tren xuong duoi.** File CSV ~805MB khong co tren GitHub — ban can **Google Drive** hoac **upload** file cleaned_data.csv.


In [ ]:
# Cell 1 — Cai thu vien
!pip install -q pandas numpy matplotlib seaborn scikit-learn statsmodels pyarrow


In [ ]:
# Cell 2 — Clone repo
import os
from pathlib import Path

REPO = "https://github.com/2274802010922/thptqg-trends.git"
ROOT = Path("/content/thptqg-trends")

if not ROOT.exists():
    !git clone https://github.com/2274802010922/thptqg-trends.git /content/thptqg-trends
else:
    !cd /content/thptqg-trends && git pull

os.chdir(ROOT)
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project:", ROOT)


## Cell 3 — Chon nguon du lieu (chi chon **1** cach)

### Cach A: Google Drive (khuyen dung cho file 805MB)


In [ ]:
# Cach A — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Sua duong dan cho dung vi tri file tren Drive cua ban
CSV_PATH = "/content/drive/MyDrive/do an thuc tap/cleaned_data.csv"
# Vi du khac: "/content/drive/MyDrive/cleaned_data.csv"

from pathlib import Path
assert Path(CSV_PATH).exists(), f"Khong tim thay: {CSV_PATH}"
print("OK:", CSV_PATH)


### Cach B: Upload truc tiep len Colab (co the mat 5-15 phut)


In [ ]:
# Cach B — Upload (bo qua neu da dung Cach A)
# from google.colab import files
# uploaded = files.upload()  # chon cleaned_data.csv
# CSV_PATH = "/content/cleaned_data.csv"

# Neu da chay Cach A o tren, bo qua cell nay.


In [ ]:
# Cell 4 — Cau hinh pham vi nam (tuy chinh neu can)
from src.config import configure

YEAR_MIN = 2021
YEAR_MAX = 2025
configure(csv_path=CSV_PATH, year_min=YEAR_MIN, year_max=YEAR_MAX)

from src.config import CSV_PATH as P
print("CSV:", P)
print("Nam:", YEAR_MIN, "-", YEAR_MAX)


In [ ]:
# Cell 5 — Chay pipeline end-to-end (~5-15 phut)
import time
from src.load_data import count_rows
from src.aggregates import save_aggregates
from src.forecast import run_forecast_pipeline
from src.plots import generate_all_figures
from src.report import generate_report
from src.config import TABLES_DIR, FIGURES_DIR, REPORTS_DIR

t0 = time.time()
print("1/5 Dem du lieu:", count_rows())
print("2/5 Aggregate...")
save_aggregates()
print("3/5 Du bao...")
print(run_forecast_pipeline().to_string(index=False))
print("4/5 Bieu do...")
for f in generate_all_figures():
    print(" ", f.name)
print("5/5 Bao cao:", generate_report())
print(f"\nXONG trong {(time.time()-t0)/60:.1f} phut")


In [ ]:
# Cell 6 — Xem bao cao
from IPython.display import Markdown, display
from pathlib import Path

report = Path("reports/BAO_CAO.md").read_text(encoding="utf-8")
display(Markdown(report))


In [ ]:
# Cell 7 — Hien thi bieu do
%matplotlib inline
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path("outputs/figures")
for p in sorted(fig_dir.glob("*.png")):
    print(p.name)
    display(Image(filename=str(p)))


In [ ]:
# Cell 8 — Tai ket qua ve may (zip)
import zipfile
from pathlib import Path
from google.colab import files

zip_path = Path("/content/thptqg_full.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["outputs", "reports"]:
        for f in Path(folder).rglob("*"):
            if f.is_file():
                z.write(f, f.as_posix())
files.download(str(zip_path))
